# Bài 41: Phát hiện đối tượng như một bài toán hồi quy

Bộ phát hiện cửa sổ trượt ở Bài 40 phân loại hàng nghìn cửa sổ kích thước cố định và gộp kết quả bằng triệt tiêu điểm không cực đại (NMS). Cách đó hoạt động được, nhưng về bản chất nó là một cách tiếp cận *phân loại* được gắn thêm vào một phép tìm kiếm — mạng không bao giờ dự đoán trực tiếp một hộp, chỉ dự đoán "có phải khuôn mặt hay không, tại đúng cửa sổ này". Các bộ phát hiện hiện đại thay vào đó coi việc định vị là một bài toán **hồi quy (regression)**: cho một ảnh, dự đoán trực tiếp tọa độ hộp bao. Bài này xây dựng phiên bản đơn giản nhất có thể của ý tưởng đó, rồi khảo sát cách các bộ phát hiện thực tế (R-CNN, YOLO, SSD) mở rộng quy mô nó.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['apollo11_crew.jpg']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.patches as patches

## Một đối tượng mỗi ảnh: dự đoán trực tiếp một hộp

Bài toán hồi quy đơn giản nhất có thể: một đốm tròn mỗi ảnh, ở vị trí và kích thước chưa biết. Thay vì một nhãn lớp, mục tiêu của mạng giờ là bốn con số — `(cx, cy, width, height)` biểu thị hộp bao, được chuẩn hóa về `[0, 1]` theo kích thước ảnh.

In [ ]:
SIZE = 32

def make_scene(rng, size=SIZE, obj_size=8):
    scene = np.zeros((size, size), dtype=np.float32)
    cx = rng.integers(obj_size, size - obj_size)
    cy = rng.integers(obj_size, size - obj_size)
    yy, xx = np.mgrid[0:size, 0:size]
    scene[((xx - cx) ** 2 + (yy - cy) ** 2) <= (obj_size * 0.5) ** 2] = 1.0
    scene = np.clip(scene + rng.normal(0, 0.05, scene.shape), 0, 1).astype(np.float32)
    box = (cx, cy, obj_size, obj_size)  # cx, cy, w, h
    return scene, box

rng = np.random.default_rng(9)
N = 400
scenes, boxes = [], []
for _ in range(N):
    s, b = make_scene(rng)
    scenes.append(s); boxes.append(b)
scenes = np.array(scenes, dtype=np.float32)
boxes = np.array(boxes, dtype=np.float32)

split = int(0.85 * N)
Xtr, Btr = scenes[:split], boxes[:split] / SIZE
Xte, Bte = scenes[split:], boxes[split:] / SIZE

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, im, b in zip(axes, Xtr[:4], boxes[:4]):
    ax.imshow(im, cmap='gray')
    ax.add_patch(patches.Rectangle((b[0] - b[2] / 2, b[1] - b[3] / 2), b[2], b[3], edgecolor='lime', facecolor='none', linewidth=2))
    ax.axis('off')
plt.show()

## Mô hình và thước đo IoU

Mạng là một mạng xương sống CNN (theo mẫu ở Bài 34) theo sau là một phần đầu hồi quy 4 đầu ra với sigmoid, để mọi dự đoán rơi vào `[0, 1]` — một tọa độ hộp chuẩn hóa hợp lệ. Nó được huấn luyện với hàm mất mát MSE đơn thuần so với hộp thực, và được đánh giá bằng **IoU** (tỉ số giao trên hợp ở Bài 40), thước đo thực sự quan trọng cho phát hiện: hộp dự đoán và hộp thực chồng lấn bao nhiêu, chứ không phải bốn con số gần nhau đến đâu khi xét riêng lẻ.

Ảnh động này minh họa công thức IoU: một hộp dự đoán trượt qua một hộp thực cố định. Vì các hộp có kích thước 40x40, phần giao (vàng) của chúng dao động từ 0 đến 1600, và phần hợp (gạch chéo) dao động từ 1600 đến 3200. Giá trị IoU thu được được vẽ khi độ lệch thay đổi.

<img src="../animations/iou_demo.gif" alt="Ảnh động minh họa tỉ số giao trên hợp: một hộp dự đoán nét đứt đỏ trượt qua một hộp thực xanh lục cố định trong khi vùng chồng lấn được tô vàng, xây dựng một đường cong hình lều của IoU theo độ lệch ngang đạt đỉnh 1.0 khi hai hộp trùng nhau và giảm về 0 khi chúng không còn chồng lấn" style="display: block; margin: 0 auto; max-width: 820px;">

Đây là code huấn luyện: lớp `Detector` định nghĩa bên dưới, được huấn luyện với hàm mất mát MSE đơn thuần so với tọa độ hộp đã chuẩn hóa trong 400 epoch, rồi được đánh giá bằng IoU trung bình trên phần kiểm tra được giữ lại.

In [ ]:
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 5, padding=2), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 5, padding=2), nn.ReLU(), nn.AdaptiveMaxPool2d(1),
        )
        self.fc = nn.Linear(32, 4)  # cx, cy, w, h, normalized

    def forward(self, x):
        return torch.sigmoid(self.fc(self.conv(x).flatten(1)))

def iou_batch(pred, target):
    pcx, pcy, pw, ph = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    tcx, tcy, tw, th = target[:, 0], target[:, 1], target[:, 2], target[:, 3]
    px0, py0, px1, py1 = pcx - pw / 2, pcy - ph / 2, pcx + pw / 2, pcy + ph / 2
    tx0, ty0, tx1, ty1 = tcx - tw / 2, tcy - th / 2, tcx + tw / 2, tcy + th / 2
    ix0, iy0 = torch.maximum(px0, tx0), torch.maximum(py0, ty0)
    ix1, iy1 = torch.minimum(px1, tx1), torch.minimum(py1, ty1)
    inter = (ix1 - ix0).clamp(min=0) * (iy1 - iy0).clamp(min=0)
    union = pw * ph + tw * th - inter
    return inter / union.clamp(min=1e-8)

torch.manual_seed(0)
model = Detector()
opt = torch.optim.Adam(model.parameters(), lr=0.005)
Xt = torch.tensor(Xtr).unsqueeze(1); Bt = torch.tensor(Btr)
for _ in range(400):
    opt.zero_grad()
    loss = F.mse_loss(model(Xt), Bt)
    loss.backward()
    opt.step()

with torch.no_grad():
    pred_te = model(torch.tensor(Xte).unsqueeze(1))
    ious = iou_batch(pred_te, torch.tensor(Bte))

print(f'mean IoU on test set: {ious.mean().item():.3f}')
print(f'fraction of test boxes with IoU > 0.5: {(ious > 0.5).float().mean().item():.1%}')

Chạy mô hình đã huấn luyện trên bốn cảnh được giữ lại, cho thấy khả năng tổng quát hóa của nó sang các ảnh mà nó chưa bao giờ thấy trong khi huấn luyện.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i, ax in enumerate(axes):
    ax.imshow(Xte[i], cmap='gray')
    tb = Bte[i] * SIZE
    pb = pred_te[i].numpy() * SIZE
    ax.add_patch(patches.Rectangle((tb[0] - tb[2] / 2, tb[1] - tb[3] / 2), tb[2], tb[3], edgecolor='lime', facecolor='none', linewidth=2, label='true'))
    ax.add_patch(patches.Rectangle((pb[0] - pb[2] / 2, pb[1] - pb[3] / 2), pb[2], pb[3], edgecolor='red', facecolor='none', linewidth=1.5, linestyle='--', label='pred'))
    ax.set_title(f'IoU={ious[i]:.2f}', fontsize=9)
    ax.axis('off')
axes[0].legend(fontsize=6, loc='upper left')
plt.show()

Ảnh động này cho thấy quá trình huấn luyện, trên cùng bốn cảnh được giữ lại. Hộp dự đoán (đỏ, nét đứt) bắt đầu từ một phỏng đoán khởi tạo ngẫu nhiên và dần khớp gần như thẳng hàng với hộp thực (xanh lục) khi huấn luyện tiến triển, với IoU của mỗi khung tăng dần theo. Các vị trí cuối cùng khớp với những gì hiển thị ở trên.

<img src="../animations/box_regression_training_demo.gif" alt="Ảnh động huấn luyện bộ hồi quy hộp: trên bốn cảnh kiểm tra được giữ lại, hộp dự đoán nét đứt đỏ bắt đầu ở xa hộp thực xanh lục và dần khớp thẳng hàng qua các epoch huấn luyện, với điểm IoU của mỗi khung tăng dần khi huấn luyện hội tụ" style="display: block; margin: 0 auto; max-width: 820px;">

## Mở rộng quy mô: hai họ bộ phát hiện thực tế

Bộ phát hiện ở trên chỉ xử lý đúng một đối tượng mỗi ảnh, vì một vector đầu ra kích thước cố định (4 con số) chỉ có thể mô tả một hộp. Nhưng cảnh thực có số lượng đối tượng thay đổi, chưa biết. Hai họ bộ phát hiện khác nhau thống trị bức tranh toàn cảnh:

**Hai giai đoạn (họ R-CNN):** đầu tiên tạo ra một số lượng vừa phải *đề xuất vùng (region proposals)* — các hộp ứng viên có khả năng chứa thứ gì đó, qua một phương pháp rẻ, không phân biệt lớp (R-CNN gốc dùng phân đoạn cổ điển; **Faster R-CNN** (<a href="../references.html#ren-2015">Ren et al., 2015</a><span class="landmark-paper">&#9733;</span>) thay vào đó học một "mạng đề xuất vùng" nhỏ) — rồi chạy một bộ-phân-loại-cộng-bộ-hồi-quy-hộp (toàn bộ kiến trúc của bài này) trên mỗi đề xuất một cách độc lập, y như chạy bộ phân loại cửa sổ trượt ở Bài 40 nhưng chỉ ở một nhúm vị trí đầy hứa hẹn thay vì mọi cửa sổ. Chính xác, nhưng chỉ nhanh đến mức (số đề xuất) x (một lượt xuôi) cho phép.

**Một giai đoạn (YOLO, SSD):** bỏ qua hoàn toàn các đề xuất. **YOLO** (You Only Look Once — Bạn Chỉ Nhìn Một Lần, <a href="../references.html#redmon-2016">Redmon et al., 2016</a><span class="landmark-paper">&#9733;</span>) chia ảnh thành một lưới thô các ô, và cho mỗi ô lưới trực tiếp dự đoán (như mạng của bài này làm) một số lượng hộp cố định cộng với một nhãn lớp cộng với một điểm số tin cậy, tất cả trong một lượt xuôi. **SSD** (Single Shot MultiBox Detector, <a href="../references.html#liu-2016-ssd">Liu et al., 2016</a><span class="landmark-paper">&#9733;</span>) theo cùng công thức một lượt nhưng dự đoán hộp từ *vài* độ phân giải bản đồ đặc trưng cùng lúc (không chỉ một lưới cuối cùng), để các lớp thô hơn tự nhiên bắt được các đối tượng lớn hơn và các lớp mịn hơn bắt được các đối tượng nhỏ hơn. Để một ô đơn lẻ có thể mô tả các đối tượng có tỉ lệ khung hình khác nhau, các bộ phát hiện một giai đoạn dùng **hộp neo (anchor boxes)**: vài hình dạng hộp định sẵn (cao, rộng, vuông) cho mỗi ô, với mạng dự đoán một *độ lệch* từ mỗi hộp neo thay vì dự đoán một hộp từ đầu. Chạy nhanh hơn, về mặt lịch sử kém chính xác hơn một chút so với các phương pháp hai giai đoạn, dù khoảng cách đã thu hẹp đáng kể.

Cả hai họ đều kết thúc với cùng một bước hậu xử lý: triệt tiêu điểm không cực đại (NMS, Bài 40) để gộp các hộp ứng viên chồng lấn mà bất kỳ cảnh nhiều đối tượng thực tế nào cũng tạo ra. (Để đơn giản, bài này bỏ qua NMS vì chỉ dự đoán một hộp bao duy nhất.)

## Trong thực tế: các bộ phát hiện thật trên ảnh thật

OpenCV cung cấp một bộ phát hiện một giai đoạn, `cv2.FaceDetectorYN` (**YuNet**, <a href="../references.html#wu-2023-yunet">Wu et al., 2023</a>), theo công thức một giai đoạn ở trên dành riêng cho khuôn mặt. Khác với chuỗi tầng Viola-Jones ở Bài 40, nó là một CNN một lượt nhỏ — cùng họ với SSD ở trên, chỉ là chuyên biệt hóa cho một lớp — và nó tự chạy NMS bên trong trước khi trả về các hộp.

In [ ]:
import urllib.request
from pathlib import Path
import cv2

CACHE_DIR = Path.home() / '.cache' / 'cvintro'
YUNET_URL = 'https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/face_detection_yunet_2023mar.onnx'
YUNET_PATH = CACHE_DIR / 'face_detection_yunet_2023mar.onnx'

def ensure_yunet():
    if YUNET_PATH.exists():
        return
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading the YuNet face detector (one-time, cached under ~/.cache/cvintro)...')
    urllib.request.urlretrieve(YUNET_URL, YUNET_PATH)

ensure_yunet()

photo = cv2.imread('../img/apollo11_crew.jpg')  # same photo Lesson 40 ran Viola-Jones on
h, w = photo.shape[:2]
yunet = cv2.FaceDetectorYN_create(str(YUNET_PATH), '', (w, h))
_, yunet_faces = yunet.detect(photo)

photo_rgb = cv2.cvtColor(photo, cv2.COLOR_BGR2RGB)
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(photo_rgb)
for x, y, fw, fh, *_, score in yunet_faces:
    ax.add_patch(patches.Rectangle((x, y), fw, fh, edgecolor='lime', facecolor='none', linewidth=2))
    ax.text(x, y - 8, f'{score:.2f}', color='lime', fontsize=9, weight='bold')
ax.set_title(f'cv2.FaceDetectorYN: {len(yunet_faces)} detections')
ax.axis('off')
plt.show()

Cả ba phi hành gia đều được tìm thấy chính xác, không có dương tính giả nào — một cải thiện so với chuỗi tầng Viola-Jones ở Bài 40. Đây là thành quả của một CNN một lượt học được so với một chuỗi tầng các đặc trưng kiểu Haar thiết kế thủ công: đặc trưng phong phú hơn, được huấn luyện đầu-cuối (end-to-end) trên dữ liệu khuôn mặt/không-phải-khuôn-mặt thật thay vì được lắp ráp từng giai đoạn.

Nhưng YuNet chỉ trả lời "có phải khuôn mặt hay không" &mdash; một lớp.

Để phát hiện tổng quát, nhiều lớp, Faster R-CNN được tiền huấn luyện trên **COCO** (<a href="../references.html#lin-2014-coco">Lin et al., 2014</a><span class="landmark-paper">&#9733;</span> — khoảng 330 nghìn ảnh chụp thật được gán nhãn trên 80 danh mục đối tượng) chỉ cách vài dòng code qua `torchvision`. Đây là cùng mẫu "mô hình tiền huấn luyện trong một dòng" như ResNet-18 ở Bài 38. Nó tự nhiên xử lý được số lượng đối tượng thay đổi, chưa biết trong mỗi ảnh — thành quả thực sự của kiến trúc đề xuất vùng đã mô tả ở trên.

In [ ]:
import urllib.request
from pathlib import Path
import cv2
import torchvision

CACHE_DIR = Path.home() / '.cache' / 'cvintro'
COCO_IMG_URL = 'http://images.cocodataset.org/val2017/000000039769.jpg'
COCO_IMG_PATH = CACHE_DIR / 'coco_sample.jpg'

def ensure_coco_sample():
    if COCO_IMG_PATH.exists():
        return
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    print('Downloading a COCO val2017 sample image (one-time, cached under ~/.cache/cvintro)...')
    urllib.request.urlretrieve(COCO_IMG_URL, COCO_IMG_PATH)

ensure_coco_sample()

weights = torchvision.models.detection.FasterRCNN_ResNet50_FPN_Weights.COCO_V1
real_detector = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights=weights)
real_detector.eval()  # frozen, no training at all -- Faster R-CNN exactly as released
coco_classes = weights.meta['categories']

img_bgr = cv2.imread(str(COCO_IMG_PATH))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_tensor = torch.tensor(img_rgb / 255.0, dtype=torch.float32).permute(2, 0, 1)

with torch.no_grad():
    result = real_detector([img_tensor])[0]

In [ ]:
score_thresh = 0.7
keep = result['scores'] > score_thresh

fig, ax = plt.subplots(figsize=(7, 5.5))
ax.imshow(img_rgb)
for box, label, score in zip(result['boxes'][keep], result['labels'][keep], result['scores'][keep]):
    x0, y0, x1, y1 = box.numpy()
    ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, edgecolor='lime', facecolor='none', linewidth=2))
    ax.text(x0, y0 - 4, f'{coco_classes[label]} {score:.2f}', color='lime', fontsize=9, weight='bold')
ax.set_title(f'Faster R-CNN pretrained on COCO ({int(keep.sum())} detections above score {score_thresh})')
ax.axis('off')
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://cocodataset.org" target="_blank" rel="noopener">tập dữ liệu COCO</a> (val2017, ảnh 000000039769)</p>

Bốn phát hiện tự tin, chính xác vượt qua ngưỡng cắt `score_thresh=0.7` — hai con mèo và hai cái điều khiển từ xa, mỗi cái trên 0.78 — dù mạng này chỉ dùng trọng số tiền huấn luyện.

Hạ `score_thresh` và chạy lại để thấy ngưỡng cắt đã che giấu những gì: vài hộp chồng lấn, độ tin cậy thấp hơn (một phỏng đoán "ghế sofa" và một phỏng đoán "giường" đều phủ gần hết ảnh, quanh điểm số 0.54). Ảnh động bên dưới thực hiện một phép quét như vậy, từ ngưỡng cắt nghiêm ngặt 0.95 xuống ngưỡng lỏng lẻo 0.05, dừng lại ở ngưỡng cắt được khuyến nghị 0.7 và ở ngưỡng cắt 0.3 được khám phá trong các bài tập bên dưới.

<img src="../animations/score_thresh_sweep_demo.gif" alt="Ảnh động quét score_thresh trên các phát hiện COCO thật của Faster R-CNN: khi ngưỡng giảm từ 0.95 xuống 0.05, nhiều hộp hơn xuất hiện, dừng lại ở 0.70 (ngưỡng cắt của sách, 4 phát hiện tự tin: hai con mèo, hai cái điều khiển) và ở 0.30 (ngưỡng cắt trong bài tập, hiện ra các phỏng đoán ghế sofa và giường độ tin cậy thấp hơn phủ gần hết ảnh)" style="display: block; margin: 0 auto; max-width: 720px;">

### Bài tập

1. Đổi `obj_size` trong `make_scene` từ giá trị cố định `8` thành một giá trị ngẫu nhiên (ví dụ `rng.integers(4, 12)`) để các đối tượng có kích thước thay đổi, rồi huấn luyện lại. IoU trung bình có giữ vững, tệ đi, hay hầu như không thay đổi — và tại sao tỉ lệ đối tượng thay đổi lại khó hơn đối với một phần đầu hồi quy kích thước cố định duy nhất so với vị trí thay đổi?
2. Hàm mất mát ở đây là MSE đơn thuần trên `(cx, cy, w, h)`, nhưng thước đo quan trọng là IoU. Hãy thay hàm mất mát bằng `1 - iou_batch(pred, target).mean()` (tối ưu trực tiếp IoU) và so sánh IoU kiểm tra trung bình cuối cùng với phiên bản huấn luyện bằng MSE. Các bộ phát hiện thực tế (ví dụ Faster R-CNN, các biến thể YOLO) làm chính xác điều này với các hàm mất mát IoU tổng quát hóa — bạn có thấy tại sao hàm mất mát MSE và thước đo IoU có thể bất đồng về việc dự đoán nào trong hai dự đoán tương tự là "tốt hơn" không?
3. Bộ phát hiện của bài này không có hộp neo và chỉ luôn xử lý một đối tượng. Hãy phác thảo (bằng lời) cách bạn sửa *đầu ra* của kiến trúc để xử lý tới 3 đối tượng mỗi ảnh dùng một lưới 3x3 trong đó mỗi ô lưới dự đoán một hộp và một độ tin cậy "có đối tượng ở đây" — ý tưởng cốt lõi đằng sau lưới của YOLO.
4. Hạ `score_thresh` trên Faster R-CNN thật xuống `0.3` và chạy lại. Đếm xem có bao nhiêu hộp xuất hiện so với ở `0.7`; những hộp mới nào thực sự là đối tượng bổ sung và những hộp nào là trùng lặp/suýt trúng trên các con mèo hay điều khiển mà NMS thường sẽ gộp lại? Sau đó hãy thử một URL ảnh COCO val2017 khác (đổi tên tệp trong `COCO_IMG_URL`, ví dụ bất kỳ `http://images.cocodataset.org/val2017/<12-digit-id>.jpg` nào) và xem mô hình tìm thấy những lớp nào.